In [ ]:
# Все библиотеки

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import ttest_ind, levene, mannwhitneyu,shapiro, \
    f_oneway, kruskal, chi2_contingency, fisher_exact
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import scikit_posthocs as sp
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import pingouin as pg
import sys
import os
from ydata_profiling import ProfileReport
from matplotlib import rcParams
from matplotlib import font_manager as fm
import matplotlib as mpl
import warnings

mpl.rcParams['font.family'] = 'HSE Sans'

warnings.filterwarnings('ignore')

# Выбросы
Если у тебя есть статистическая задача, в которое ты использушь неробастные оценкки (на которые влияют выбросы), например, среднее или T-test.

**Проверь выбросы!**

Сначала визуально boxplot, если они визульно есть, то стоит их отбросить по IQR или тп.

**Обращай внимание на смысл данных** не может быть -5 апельсинов.

In [1]:
import pandas as pd

def IQR_outliers_remove(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for column in columns:
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        df = df[(df[column] >= Q1 - 1.5 * IQR) & (df[column] <= Q3 + 1.5 * IQR)]
    return df

# Корреляция

1. Проверка типов переменных
- В функу передаем categorial - список категориальных переменных.
- проверка нормальности - тест Шапиро

2. Выбор метрики связи
 - числовые-числовые: корреляция (pearson/spearman) в [-1,1]
 - категориальные-категориальные: Cramér's V в [0,1]
 - числовые-категориальные: eta-squared (ANOVA effect size) в [0,1]

<img src="Тип%20корреляции.jpg">

In [ ]:
# Универсальная функцию для построения трёх heatmap'ов для смешанных типов данных.
# - числовые-числовые: корреляция (pearson/spearman) в [-1,1]
# - категориальные-категориальные: Cramér's V в [0,1]
# - числовые-категориальные: eta-squared (ANOVA effect size) в [0,1]
#
# Как интерпретировать резульаты:
# - pearson/spearman или eta-squared -> 0.1/0.3/0.5/0.8 -> слабая/средняя/сильная/оч сильная
# - Cramér's V -> 0.01/0.06/0.14 -> слабая/средняя/сильная
#
# Функция отображает три отдельных графика (по одному на каждую метрику) и возвращает
# соответствующие DataFrame'ы.
#

from typing import List, Dict, Optional, Any

def _cramers_v(x: pd.Series, y: pd.Series) -> float:
    """Вычислить Cramér's V для двух категориальных Series."""
    # Убрать пары с пропусками
    mask = x.notna() & y.notna()
    x2 = x[mask]
    y2 = y[mask]
    if x2.size == 0:
        return np.nan
    table = pd.crosstab(x2, y2)
    if table.size == 0:
        return np.nan
    n = table.values.sum()
    # Если одна из осей имеет только 1 уровень, V = 0 (нет информации)
    r, c = table.shape
    k = min(r - 1, c - 1)
    if k <= 0 or n == 0:
        return 0.0
    chi2, p, dof, expected = chi2_contingency(table, correction=False)
    return float(np.sqrt(chi2 / (n * k)))

def _eta_squared(numeric: pd.Series, categ: pd.Series) -> float:
    """Вычислить eta-squared (отношение межгрупповой дисперсии к общей)."""
    mask = numeric.notna() & categ.notna()
    x = numeric[mask]
    g = categ[mask]
    if x.size == 0:
        return np.nan
    overall_mean = x.mean()
    groups = x.groupby(g)
    ss_between = ((groups.size() * (groups.mean() - overall_mean) ** 2).sum())
    ss_total = ((x - overall_mean) ** 2).sum()
    if ss_total == 0:
        return 0.0
    return float(ss_between / ss_total)

def mixed_corr_heatmaps(
    df: pd.DataFrame,
    categorical_cols: Optional[List[str]] = None,
    method_num: str = "pearson",
    decimals: int = 2,
    figsize_each: tuple = (6,6),
    show: bool = True
) -> Dict[str, Any]:
    """
    Построить три heatmap'а для смешанных данных (числовые и категориальные).

    Параметры
    ----------
    df : pd.DataFrame
        Входной датафрейм.
    categorical_cols : list[str], optional
        Список имен столбцов, которые считаем категориальными. Если None —
        будут автоматически выбраны столбцы типов object/category/bool.
    method_num : {'pearson','spearman'}
        Метод корреляции для числовых-числовых пар.
    decimals : int
        Сколько знаков отображать в аннотациях клеток.
    figsize_each : tuple
        Размер каждой фигуры (одна фигура = одна heatmap).
    show : bool
        Показывать графики сразу (True) или только возвращать объекты (False).

    Возвращает
    -------
    dict : {
        'num_num': pd.DataFrame,  # корреляции числовых (слова столбцы/индексы)
        'cat_cat': pd.DataFrame,  # Cramér's V (0..1)
        'num_cat': pd.DataFrame,  # eta-squared (0..1) — строки = числовые, колонки = категориальные
        'figs': list(matplotlib.figure.Figure)
    }

    Примечания
    ---------
    - Cramér's V и eta-squared не имеют знака — показывают только силу связи.
    - Для числовых пар можно получить отрицательные значения (напр., Pearson).
    - Функция создаёт по одной отдельной фигуре для каждой матрицы (чтобы избежать наложения).
    """
    # Проверки и подготовка списков колонок
    if categorical_cols is None:
        categorical_cols = df.select_dtypes(include=['object','category','bool']).columns.tolist()
    else:
        # Оставляем только реально существующие в df
        categorical_cols = [c for c in categorical_cols if c in df.columns]
    numeric_cols = [c for c in df.columns if c not in categorical_cols]

    # 1) Числовые-числовые
    if len(numeric_cols) > 0:
        corr_num = pd.DataFrame(index=numeric_cols, columns=numeric_cols, dtype=float)
        for i in numeric_cols:
            for j in numeric_cols:
                if i == j:
                    corr_num.loc[i, j] = 1.0
                else:
                    v1 = df[i].corr(df[j], method='spearman')
                    v2 = df[i].corr(df[j], method='pearson')
                    corr_num.loc[i, j] = v1 if abs(v1) > abs(v2) else v2

    else:
        corr_num = pd.DataFrame(index=[], columns=[])

    # 2) Категориальные-категориальные (Cramér's V)
    if len(categorical_cols) > 0:
        cramers = pd.DataFrame(index=categorical_cols, columns=categorical_cols, dtype=float)
        for i in categorical_cols:
            for j in categorical_cols:
                if i == j:
                    cramers.at[i, j] = 1.0
                else:
                    cramers.at[i, j] = _cramers_v(df[i], df[j])
        cramers = cramers.astype(float)
    else:
        cramers = pd.DataFrame(index=[], columns=[])

    # 3) Числовые-категориальные (eta-squared)
    if len(numeric_cols) > 0 and len(categorical_cols) > 0:
        eta = pd.DataFrame(index=numeric_cols, columns=categorical_cols, dtype=float)
        for num in numeric_cols:
            for cat in categorical_cols:
                eta.at[num, cat] = _eta_squared(df[num], df[cat])
        eta = eta.astype(float)
    else:
        eta = pd.DataFrame(index=numeric_cols, columns=categorical_cols)

    figs = []
    # Визуализация: отдельная фигура на каждую матрицу (требование)
    # 1) Числовая корреляция (диапазон -1..1)
    if corr_num.size != 0:
        fig1 = plt.figure(figsize=figsize_each)
        ax1 = fig1.add_subplot(1,1,1)
        im1 = ax1.imshow(corr_num.values, vmin=-1, vmax=1, aspect='auto', interpolation='nearest')
        ax1.set_xticks(np.arange(len(corr_num.columns)))
        ax1.set_yticks(np.arange(len(corr_num.index)))
        ax1.set_xticklabels(corr_num.columns, rotation=90)
        ax1.set_yticklabels(corr_num.index)
        ax1.set_title(f'Числовые vs Числовые ({method_num})')
        # Аннотации
        for i in range(corr_num.shape[0]):
            for j in range(corr_num.shape[1]):
                val = corr_num.iat[i,j]
                if pd.isna(val):
                    txt = 'NaN'
                else:
                    txt = f'{val:.{decimals}f}'
                ax1.text(j, i, txt, ha='center', va='center', fontsize=8)
        fig1.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
        figs.append(fig1)

    # 2) Категории vs Категории (0..1)
    if cramers.size != 0:
        fig2 = plt.figure(figsize=figsize_each)
        ax2 = fig2.add_subplot(1,1,1)
        im2 = ax2.imshow(cramers.values, vmin=0, vmax=1, aspect='auto', interpolation='nearest')
        ax2.set_xticks(np.arange(len(cramers.columns)))
        ax2.set_yticks(np.arange(len(cramers.index)))
        ax2.set_xticklabels(cramers.columns, rotation=90)
        ax2.set_yticklabels(cramers.index)
        ax2.set_title("Категориальные vs Категориальные (Cramér's V)")
        for i in range(cramers.shape[0]):
            for j in range(cramers.shape[1]):
                val = cramers.iat[i,j]
                if pd.isna(val):
                    txt = 'NaN'
                else:
                    txt = f'{val:.{decimals}f}'
                ax2.text(j, i, txt, ha='center', va='center', fontsize=8)
        fig2.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
        figs.append(fig2)

    # 3) Числовые vs Категориальные (eta-squared 0..1)
    if eta.size != 0:
        fig3 = plt.figure(figsize=figsize_each)
        ax3 = fig3.add_subplot(1,1,1)
        im3 = ax3.imshow(eta.values, vmin=0, vmax=1, aspect='auto', interpolation='nearest')
        ax3.set_xticks(np.arange(len(eta.columns)))
        ax3.set_yticks(np.arange(len(eta.index)))
        ax3.set_xticklabels(eta.columns, rotation=90)
        ax3.set_yticklabels(eta.index)
        ax3.set_title("Числовые vs Категориальные (eta-squared)")
        for i in range(eta.shape[0]):
            for j in range(eta.shape[1]):
                val = eta.iat[i,j]
                if pd.isna(val):
                    txt = 'NaN'
                else:
                    txt = f'{val:.{decimals}f}'
                ax3.text(j, i, txt, ha='center', va='center', fontsize=8)
        fig3.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)
        figs.append(fig3)

    if show:
        # Показать фигуры (в Jupyter они появятся под выполнением этой ячейки)
        for f in figs:
            display(f)

    # Возвращаем DataFrame'ы и фигуры
    return {
        'num_num': corr_num,
        'cat_cat': cramers,
        'num_cat': eta,
        'figs': figs
    }

# Графики пропусков

**Если нужно графиком показать кол-во пропусков**

In [ ]:
import missingno as msno

msno.bar(df)

**Корреляция между пропусками**

In [ ]:
msno.heatmap(df)

# Нормальность

Изучая нормальность данных нужно показать и гистограмму, и боксплот.
В `base.ipynp` есть функция `norm()`, которая выводит нормально ли распределены данные.

ЦПТ освобождает от проверки нормальности во всех тестах, где ключевой параметр — среднее или сумма, если выборка большая и данные независимы с конечной дисперсией.(Z-test: Одновыборочный, Двухвыборочный, для пропорций; t-тесты Стьюдента: Одновыборочный, Парный, Двухвыборочный; ANOVA: Однофакторный и многофакторный и другие)

<img src="Предобработка_1.png">
<img src="Предобработка_2.png">

# Максимум в задаче с моделированием
Если в задаче про моделирование есть пункт **найдите максимум**, то ты должен найти оптимальные параметры задачи, а потом с ними с моделировть лучший вариант из всех возможных

# Указание параметров, генерирующих распределения
По опыту прошлого года за это рубили баллы, поэтому если у вас есть какое-то сгенерированое вами распределение, то не поленитесь и покажите параметры, которые его задают.

# Читаемые графики
Указывайте заголовок, название осей, x-lim, y-lim. 

**Проверяйте свой график на "читаемость"**

# Задача на симуляцию. Ошибка с округлением
В задаче на симуляцию после того как вы получили оптимум, прогоните еще раз но с сильно меньшим диапазоном, например, +- 5, но с сильно большим количеством итераций и меньшим шагом, чтобы точно не ошибиться

# Задача на оценку
Где просят оценить интервал, подставлять максимальные и минимальные значения из условия